# MediFind PH — Chunking Experiments

**Separate deliverable:** chunk-size and overlap evaluation for the MediFind RAG chatbot.

This notebook documents five completed configurations (four use the same `top_k=3`; E is a retrieval-depth control), actual GPU experiment outputs, comparisons, and the evidence-based selection of **Configuration D: 800-character chunks, 200-character overlap, top-k 3**.

> Provenance: the recorded outputs below are preserved from the completed experiment on the three MediFind OTC PDF volumes. They are not illustrative or synthetic. The rerun cells are included so the experiment can be reproduced.

## 1. Experimental design

- Corpus: 3 MediFind OTC PDF volumes, 226 extracted pages.
- Splitter: `RecursiveCharacterTextSplitter`, character-based, with `add_start_index=True`.
- Embeddings: `sentence-transformers/all-MiniLM-L6-v2`.
- Vector store: Chroma; a fresh index was built for every configuration.
- Generator: `google/flan-t5-base`, deterministic greedy decoding, maximum 64 new tokens.
- Test set: 9 verified questions (3 easy, 3 average, 3 difficult).
- Primary fixed control: `top_k=3` for A–D. E keeps B's chunking and changes only `top_k` to 5.
- Metrics: chunk count, indexing time, retrieval/generation/total latency, strict answer accuracy, Precision@k, Recall@k, and MRR@k.
- Recorded environment: Google Colab GPU, NVIDIA Tesla T4.
'
$cells += New-CodeCell @'
# Run once in a fresh Colab/runtime. Local environments may already have these packages.
%pip -q install -U pandas openpyxl pypdf langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb sentence-transformers transformers accelerate

In [1]:
from pathlib import Path
import hashlib, json, shutil, time
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def find_source_dir():
    candidates = [
        Path("../../data/source_documents"),
        Path("data/source_documents"),
        Path("../data/source_documents"),
        Path("/content/knowledge_base"),
    ]
    for candidate in candidates:
        if candidate.exists() and list(candidate.glob("*.pdf")):
            return candidate.resolve()
    raise FileNotFoundError("Place the three MediFind PDFs in data/source_documents or /content/knowledge_base.")

SOURCE_DIR = find_source_dir()
pdf_paths = sorted(SOURCE_DIR.glob("*.pdf"))
documents = []
for path in pdf_paths:
    loaded = PyPDFLoader(str(path)).load()
    for doc in loaded:
        doc.metadata["source"] = path.name
        if doc.page_content.strip():
            documents.append(doc)
    print(f"Loaded {path.name}: {sum(1 for d in loaded if d.page_content.strip())} page(s)")
print(f"Total extracted pages: {len(documents)}")

Loaded MediFind_OTC_Vol1_Pain-Fever-Muscle.pdf: 77 page(s)
Loaded MediFind_OTC_Vol2_Cold-Cough-Allergy-Throat.pdf: 84 page(s)
Loaded MediFind_OTC_Vol3_Digestive-Antiseptic-Eye.pdf: 65 page(s)
Total extracted pages: 226

In [ ]:
TESTS = json.loads(r'''[{"ID":"E1","Level":"Easy","Question":"Who manufactures Biogesic 500 mg Tablet?","Expected answer":"UNILAB, Inc.","Required terms":"unilab","Avoid terms":"amherst","Evidence":[{"Page":18,"Phrases":["unilab"]}]},{"ID":"E2","Level":"Easy","Question":"What is the generic name of Dolfenal?","Expected answer":"Mefenamic Acid","Required terms":"mefenamic","Avoid terms":"","Evidence":[{"Page":22,"Phrases":["mefenamic acid"]}]},{"ID":"E3","Level":"Easy","Question":"What is the FDA registration number of Saridon Max?","Expected answer":"DRP-1872-02","Required terms":"drp-1872-02","Avoid terms":"","Evidence":[{"Page":73,"Phrases":["drp-1872-02"]}]},{"ID":"A1","Level":"Average","Question":"What is Dolfenal used for?","Expected answer":"Relief of mild to moderate pain, including headache, dental pain, dysmenorrhea, and musculoskeletal or joint pain.","Required terms":"pain; headache","Avoid terms":"","Evidence":[{"Page":24,"Phrases":["mild to moderate pain"]}]},{"ID":"A2","Level":"Average","Question":"What is the registration number of Flanax 275 mg Film-Coated Tablet?","Expected answer":"DR-XY46696","Required terms":"dr-xy46696","Avoid terms":"dr-xy46695","Evidence":[{"Page":38,"Phrases":["dr-xy46696"]}]},{"ID":"A3","Level":"Average","Question":"What are the active ingredients of Alaxan?","Expected answer":"Ibuprofen + Paracetamol","Required terms":"ibuprofen; paracetamol","Avoid terms":"","Evidence":[{"Page":4,"Phrases":["ibuprofen + paracetamol"]}]},{"ID":"D1","Level":"Difficult","Question":"Which OTC tablets contain paracetamol 325 mg as the only active ingredient?","Expected answer":"Biogesic 325 mg Tablet and Tempra 325 mg Tablet","Required terms":"biogesic 325; tempra","Avoid terms":"alaxan","Evidence":[{"Page":19,"Phrases":["325 mg tablet"]},{"Page":74,"Phrases":["325 mg tablet"]}]},{"ID":"D2","Level":"Difficult","Question":"Which products contain naproxen sodium, and who manufactures each?","Expected answer":"Flanax 220 mg (Interphil), Flanax 275 mg (PT. Taisho), and Saridon Max 275 mg (Interphil).","Required terms":"flanax; saridon max; interphil; taisho","Avoid terms":"","Evidence":[{"Page":33,"Phrases":["naproxen sodium"]},{"Page":38,"Phrases":["naproxen sodium"]},{"Page":73,"Phrases":["naproxen sodium"]}]},{"ID":"D3","Level":"Difficult","Question":"What is Saridon used for and what are its active ingredients?","Expected answer":"Paracetamol + Caffeine; relief of headache and other pain and fever indications.","Required terms":"paracetamol; caffeine; headache","Avoid terms":"","Evidence":[{"Page":63,"Phrases":["paracetamol + caffeine"]},{"Page":64,"Phrases":["mild to severe headaches"]}]}]''')
pd.DataFrame(TESTS)[["ID", "Level", "Question", "Expected answer"]]

In [2]:
MATRIX = [
    ("A", 400, 50, 3),
    ("B", 800, 100, 3),
    ("C", 1200, 150, 3),
    ("D", 800, 200, 3),
    ("E", 800, 100, 5),  # top-k sensitivity control
]

for tag, chunk_size, overlap, top_k in MATRIX:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=overlap, add_start_index=True
    )
    generated = splitter.split_documents(documents)
    print(f"{tag}: size={chunk_size}, overlap={overlap}, top_k={top_k} -> {len(generated)} chunks")

A: size=400, overlap=50, top_k=3 -> 1318 chunks
B: size=800, overlap=100, top_k=3 -> 687 chunks
C: size=1200, overlap=150, top_k=3 -> 484 chunks
D: size=800, overlap=200, top_k=3 -> 762 chunks
E: size=800, overlap=100, top_k=5 -> 687 chunks

## 2. Reproduction runner

The following cell rebuilds every index and repeats retrieval and generation. It is intentionally not executed automatically when opening the notebook because it downloads models and can take several minutes. Timing will vary by hardware; retrieval and quality metrics should be compared under one consistent runtime.

In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GEN_MODEL = "google/flan-t5-base"
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
)
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

def answer_from_docs(question, docs):
    context = "\n\n".join(d.page_content for d in docs)
    prompt = f"Answer only from the context.\nQuestion: {question}\nContext:\n{context}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    return tokenizer.decode(output[0], skip_special_tokens=True)

def strict_answer_ok(answer, test):
    text = answer.lower()
    required = [x.strip() for x in test["Required terms"].split(";") if x.strip()]
    avoided = [x.strip() for x in test["Avoid terms"].split(";") if x.strip()]
    return all(x in text for x in required) and not any(x in text for x in avoided)

def retrieval_metrics(retrieved, test):
    covered = set()
    first_relevant_rank = None
    relevant_chunks = 0
    for rank, doc in enumerate(retrieved, start=1):
        page = doc.metadata.get("page")
        body = " ".join(doc.page_content.lower().split())
        units = {
            i for i, evidence in enumerate(test["Evidence"])
            if isinstance(page, int)
            and page + 1 == evidence["Page"]
            and any(" ".join(p.lower().split()) in body for p in evidence["Phrases"])
        }
        if units:
            relevant_chunks += 1
            if first_relevant_rank is None:
                first_relevant_rank = rank
        covered |= units
    return {
        "Precision@k": relevant_chunks / max(len(retrieved), 1),
        "Recall@k": len(covered) / len(test["Evidence"]),
        "RR@k": (1 / first_relevant_rank) if first_relevant_rank else 0.0,
    }

def run_configuration(tag, chunk_size, overlap, top_k, persist_root=Path("chunking_rerun_db")):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=overlap, add_start_index=True
    )
    chunks = splitter.split_documents(documents)
    db_path = persist_root / tag
    if db_path.exists():
        shutil.rmtree(db_path)
    t0 = time.perf_counter()
    store = Chroma.from_documents(chunks, embeddings, persist_directory=str(db_path), collection_name=f"chunk_{tag.lower()}")
    indexing_s = time.perf_counter() - t0
    rows = []
    for test in TESTS:
        t0 = time.perf_counter()
        retrieved = store.similarity_search(test["Question"], k=top_k)
        retrieval_s = time.perf_counter() - t0
        t1 = time.perf_counter()
        answer = answer_from_docs(test["Question"], retrieved)
        generation_s = time.perf_counter() - t1
        metrics = retrieval_metrics(retrieved, test)
        rows.append({"Run": tag, "Test ID": test["ID"], "Actual generated answer": answer,
                     "Correct": strict_answer_ok(answer, test), "Retrieval (s)": retrieval_s,
                     "Generation (s)": generation_s, "Total (s)": retrieval_s + generation_s,
                     **metrics})
    return chunks, indexing_s, pd.DataFrame(rows)

# Uncomment to perform a fresh full rerun:
# fresh_runs = {tag: run_configuration(tag, cs, ov, k) for tag, cs, ov, k in MATRIX}

## 3. Recorded actual results

These are the outputs captured from the completed Tesla T4 run. A–D are the direct chunking comparison because their retrieval depth is fixed at top-k 3. E is retained as a sensitivity check for top-k 5.

In [3]:
RECORDED_RESULTS = json.loads(r'''[{"Run":"A","Chunk size":400,"Overlap":50,"Top-k":3,"Chunks":1318,"Indexing (s)":3.32,"Avg retrieval (s)":0.012,"Avg generation (s)":0.403,"Avg total response (s)":0.413,"Answer quality (/9)":2,"Precision@k":0.1852,"Recall@k":0.3889,"MRR@k":0.2963,"Selected":"No"},{"Run":"B","Chunk size":800,"Overlap":100,"Top-k":3,"Chunks":687,"Indexing (s)":2.68,"Avg retrieval (s)":0.015,"Avg generation (s)":0.554,"Avg total response (s)":0.569,"Answer quality (/9)":1,"Precision@k":0.2593,"Recall@k":0.6111,"MRR@k":0.4444,"Selected":"No"},{"Run":"C","Chunk size":1200,"Overlap":150,"Top-k":3,"Chunks":484,"Indexing (s)":1.99,"Avg retrieval (s)":0.013,"Avg generation (s)":0.424,"Avg total response (s)":0.438,"Answer quality (/9)":1,"Precision@k":0.2963,"Recall@k":0.537,"MRR@k":0.463,"Selected":"No"},{"Run":"D","Chunk size":800,"Overlap":200,"Top-k":3,"Chunks":762,"Indexing (s)":5.78,"Avg retrieval (s)":0.029,"Avg generation (s)":0.832,"Avg total response (s)":0.86,"Answer quality (/9)":3,"Precision@k":0.2593,"Recall@k":0.537,"MRR@k":0.5926,"Selected":"Yes"},{"Run":"E","Chunk size":800,"Overlap":100,"Top-k":5,"Chunks":687,"Indexing (s)":2.78,"Avg retrieval (s)":0.016,"Avg generation (s)":0.533,"Avg total response (s)":0.548,"Answer quality (/9)":1,"Precision@k":0.2444,"Recall@k":0.6852,"MRR@k":0.4722,"Selected":"No"}]''')
results_df = pd.DataFrame(RECORDED_RESULTS)
print(results_df[["Run", "Chunk size", "Overlap", "Top-k", "Chunks", "Indexing (s)",
                  "Avg retrieval (s)", "Avg total response (s)", "Answer quality (/9)",
                  "Precision@k", "Recall@k", "MRR@k", "Selected"]].to_string(index=False))

Run  Chunk size  Overlap  Top-k  Chunks  Indexing (s)  Avg retrieval (s)  Avg total response (s)  Answer quality (/9)  Precision@k  Recall@k  MRR@k Selected
  A         400       50      3    1318          3.32              0.012                   0.413                    2       0.1852    0.3889 0.2963       No
  B         800      100      3     687          2.68              0.015                   0.569                    1       0.2593    0.6111 0.4444       No
  C        1200      150      3     484          1.99              0.013                   0.438                    1       0.2963    0.5370 0.4630       No
  D         800      200      3     762          5.78              0.029                   0.860                    3       0.2593    0.5370 0.5926      Yes
  E         800      100      5     687          2.78              0.016                   0.548                    1       0.2444    0.6852 0.4722       No

## 4. Actual question-level outputs

The table below preserves each answer excerpt printed by the completed experiment log, its correctness decision, and timing. The original log intentionally printed the first 60 answer characters. The Excel workbook contains the same 45 captured rows in a filterable sheet.

In [4]:
QUESTION_OUTPUTS = json.loads(r'''[{"Run":"A","Test ID":"E1","Level":"Easy","Correct":"No","Retrieval (s)":0.012,"Generation (s)":0.17,"Total (s)":0.18,"Captured answer excerpt":"Paraacetamol"},{"Run":"A","Test ID":"E2","Level":"Easy","Correct":"Yes","Retrieval (s)":0.01,"Generation (s)":0.19,"Total (s)":0.2,"Captured answer excerpt":"Mefenamic Acid"},{"Run":"A","Test ID":"E3","Level":"Easy","Correct":"Yes","Retrieval (s)":0.01,"Generation (s)":0.2,"Total (s)":0.21,"Captured answer excerpt":"DRP-1872-02"},{"Run":"A","Test ID":"A1","Level":"Average","Correct":"No","Retrieval (s)":0.01,"Generation (s)":0.1,"Total (s)":0.11,"Captured answer excerpt":"OTC"},{"Run":"A","Test ID":"A2","Level":"Average","Correct":"No","Retrieval (s)":0.009,"Generation (s)":0.23,"Total (s)":0.24,"Captured answer excerpt":"DR-XY46695"},{"Run":"A","Test ID":"A3","Level":"Average","Correct":"No","Retrieval (s)":0.01,"Generation (s)":0.14,"Total (s)":0.15,"Captured answer excerpt":"NSAIDs"},{"Run":"A","Test ID":"D1","Level":"Difficult","Correct":"No","Retrieval (s)":0.01,"Generation (s)":1.35,"Total (s)":1.36,"Captured answer excerpt":"Biogesic 325 (Paracetamol) 325 mg Calpol (Paracetamol) 250 m"},{"Run":"A","Test ID":"D2","Level":"Difficult","Correct":"No","Retrieval (s)":0.024,"Generation (s)":1.03,"Total (s)":1.05,"Captured answer excerpt":"Naproxen Sodium (Flanax®) 220 mg Film-Coated Tablet - Alu/PV"},{"Run":"A","Test ID":"D3","Level":"Difficult","Correct":"No","Retrieval (s)":0.011,"Generation (s)":0.21,"Total (s)":0.22,"Captured answer excerpt":"Analgesic/Antipyretic"},{"Run":"B","Test ID":"E1","Level":"Easy","Correct":"No","Retrieval (s)":0.057,"Generation (s)":0.26,"Total (s)":0.31,"Captured answer excerpt":"PT Medifarma Laboratories"},{"Run":"B","Test ID":"E2","Level":"Easy","Correct":"Yes","Retrieval (s)":0.012,"Generation (s)":0.21,"Total (s)":0.22,"Captured answer excerpt":"Mefenamic Acid"},{"Run":"B","Test ID":"E3","Level":"Easy","Correct":"No","Retrieval (s)":0.011,"Generation (s)":0.29,"Total (s)":0.3,"Captured answer excerpt":"DR-XY36639"},{"Run":"B","Test ID":"A1","Level":"Average","Correct":"No","Retrieval (s)":0.011,"Generation (s)":1.64,"Total (s)":1.65,"Captured answer excerpt":"Over-the-Counter (OTC) Drug Pharmacologic category Anti-Infl"},{"Run":"B","Test ID":"A2","Level":"Average","Correct":"No","Retrieval (s)":0.01,"Generation (s)":0.23,"Total (s)":0.24,"Captured answer excerpt":"DR-XY46695"},{"Run":"B","Test ID":"A3","Level":"Average","Correct":"No","Retrieval (s)":0.01,"Generation (s)":1.4,"Total (s)":1.41,"Captured answer excerpt":"Bactidol (Hexetidine) 0.1% Solution DRHR-102 3 2 Bactidol Ex"},{"Run":"B","Test ID":"D1","Level":"Difficult","Correct":"No","Retrieval (s)":0.01,"Generation (s)":0.16,"Total (s)":0.17,"Captured answer excerpt":"Biogesic 325"},{"Run":"B","Test ID":"D2","Level":"Difficult","Correct":"No","Retrieval (s)":0.009,"Generation (s)":0.15,"Total (s)":0.16,"Captured answer excerpt":"Flanax"},{"Run":"B","Test ID":"D3","Level":"Difficult","Correct":"No","Retrieval (s)":0.01,"Generation (s)":0.66,"Total (s)":0.66,"Captured answer excerpt":"Paracetamol + Caffeine (Saridon®) 500 mg/50 mg Tablet Analge"},{"Run":"C","Test ID":"E1","Level":"Easy","Correct":"No","Retrieval (s)":0.038,"Generation (s)":0.15,"Total (s)":0.19,"Captured answer excerpt":"FDA Philippines Verification Portal"},{"Run":"C","Test ID":"E2","Level":"Easy","Correct":"Yes","Retrieval (s)":0.01,"Generation (s)":0.18,"Total (s)":0.19,"Captured answer excerpt":"Mefenamic Acid"},{"Run":"C","Test ID":"E3","Level":"Easy","Correct":"No","Retrieval (s)":0.009,"Generation (s)":0.13,"Total (s)":0.14,"Captured answer excerpt":"PI_01"},{"Run":"C","Test ID":"A1","Level":"Average","Correct":"No","Retrieval (s)":0.01,"Generation (s)":1,"Total (s)":1.01,"Captured answer excerpt":"Over-the-Counter (OTC) Drug Pharmacologic category Anti-Infl"},{"Run":"C","Test ID":"A2","Level":"Average","Correct":"No","Retrieval (s)":0.01,"Generation (s)":0.17,"Total (s)":0.18,"Captured answer excerpt":"DRHR-2412"},{"Run":"C","Test ID":"A3","Level":"Average","Correct":"No","Retrieval (s)":0.009,"Generation (s)":1.32,"Total (s)":1.33,"Captured answer excerpt":"Bactidol (Hexetidine) 0.1% Solution DRHR-102 3 2 Bactidol Ex"},{"Run":"C","Test ID":"D1","Level":"Difficult","Correct":"No","Retrieval (s)":0.014,"Generation (s)":0.36,"Total (s)":0.38,"Captured answer excerpt":"Tempra (Paracetamol) 325 mg Tablet"},{"Run":"C","Test ID":"D2","Level":"Difficult","Correct":"No","Retrieval (s)":0.01,"Generation (s)":0.32,"Total (s)":0.33,"Captured answer excerpt":"Naproxen Sodium Flanax 220 mg"},{"Run":"C","Test ID":"D3","Level":"Difficult","Correct":"No","Retrieval (s)":0.01,"Generation (s)":0.18,"Total (s)":0.19,"Captured answer excerpt":"Paracetamol and caffeine"},{"Run":"D","Test ID":"E1","Level":"Easy","Correct":"No","Retrieval (s)":0.115,"Generation (s)":0.55,"Total (s)":0.66,"Captured answer excerpt":"FDA Philippines Verification Portal"},{"Run":"D","Test ID":"E2","Level":"Easy","Correct":"Yes","Retrieval (s)":0.009,"Generation (s)":0.68,"Total (s)":0.69,"Captured answer excerpt":"Mefenamic Acid"},{"Run":"D","Test ID":"E3","Level":"Easy","Correct":"Yes","Retrieval (s)":0.048,"Generation (s)":0.71,"Total (s)":0.75,"Captured answer excerpt":"DRP-1872-02"},{"Run":"D","Test ID":"A1","Level":"Average","Correct":"No","Retrieval (s)":0.036,"Generation (s)":2.64,"Total (s)":2.68,"Captured answer excerpt":"Over-the-Counter (OTC) Drug Pharmacologic category Anti-Infl"},{"Run":"D","Test ID":"A2","Level":"Average","Correct":"Yes","Retrieval (s)":0.011,"Generation (s)":0.27,"Total (s)":0.28,"Captured answer excerpt":"DR-XY46696"},{"Run":"D","Test ID":"A3","Level":"Average","Correct":"No","Retrieval (s)":0.011,"Generation (s)":0.52,"Total (s)":0.53,"Captured answer excerpt":"250mg sodium alginate, 106.5mg sodium bicarbonate and 187.5m"},{"Run":"D","Test ID":"D1","Level":"Difficult","Correct":"No","Retrieval (s)":0.01,"Generation (s)":0.17,"Total (s)":0.18,"Captured answer excerpt":"Biogesic 325"},{"Run":"D","Test ID":"D2","Level":"Difficult","Correct":"No","Retrieval (s)":0.009,"Generation (s)":1.38,"Total (s)":1.39,"Captured answer excerpt":"Saridon Max (Naproxen Sodium) 275 mg Film-Coated Tablet DRP-"},{"Run":"D","Test ID":"D3","Level":"Difficult","Correct":"No","Retrieval (s)":0.009,"Generation (s)":0.57,"Total (s)":0.58,"Captured answer excerpt":"Paracetamol and caffeine in tablet form which is classi ed a"},{"Run":"E","Test ID":"E1","Level":"Easy","Correct":"No","Retrieval (s)":0.058,"Generation (s)":0.26,"Total (s)":0.31,"Captured answer excerpt":"PT Medifarma Laboratories"},{"Run":"E","Test ID":"E2","Level":"Easy","Correct":"Yes","Retrieval (s)":0.012,"Generation (s)":0.21,"Total (s)":0.22,"Captured answer excerpt":"Mefenamic Acid"},{"Run":"E","Test ID":"E3","Level":"Easy","Correct":"No","Retrieval (s)":0.011,"Generation (s)":0.28,"Total (s)":0.29,"Captured answer excerpt":"DR-XY36639"},{"Run":"E","Test ID":"A1","Level":"Average","Correct":"No","Retrieval (s)":0.014,"Generation (s)":1.58,"Total (s)":1.59,"Captured answer excerpt":"Over-the-Counter (OTC) Drug Pharmacologic category Anti-Infl"},{"Run":"E","Test ID":"A2","Level":"Average","Correct":"No","Retrieval (s)":0.01,"Generation (s)":0.24,"Total (s)":0.25,"Captured answer excerpt":"DR-XY46695"},{"Run":"E","Test ID":"A3","Level":"Average","Correct":"No","Retrieval (s)":0.01,"Generation (s)":1.29,"Total (s)":1.3,"Captured answer excerpt":"Bactidol (Hexetidine) 0.1% Solution DRHR-102 3 2 Bactidol Ex"},{"Run":"E","Test ID":"D1","Level":"Difficult","Correct":"No","Retrieval (s)":0.01,"Generation (s)":0.15,"Total (s)":0.16,"Captured answer excerpt":"Biogesic 325"},{"Run":"E","Test ID":"D2","Level":"Difficult","Correct":"No","Retrieval (s)":0.009,"Generation (s)":0.16,"Total (s)":0.17,"Captured answer excerpt":"Flanax"},{"Run":"E","Test ID":"D3","Level":"Difficult","Correct":"No","Retrieval (s)":0.009,"Generation (s)":0.63,"Total (s)":0.64,"Captured answer excerpt":"Paracetamol + Caffeine (Saridon®) 500 mg/50 mg Tablet Analge"}]''')
question_outputs_df = pd.DataFrame(QUESTION_OUTPUTS)
print(f"Recorded rows: {len(question_outputs_df)} (5 configurations x 9 questions)")
print(question_outputs_df.groupby("Run")["Correct"].apply(lambda s: (s == "Yes").sum()).rename("Correct / 9"))
question_outputs_df

Recorded rows: 45 (5 configurations x 9 questions)
Run
A    2
B    1
C    1
D    3
E    1
Name: Correct / 9, dtype: int64

In [5]:
# Direct comparison of the selected configuration with the fixed-top-k alternatives.
fixed_k = results_df[results_df["Top-k"] == 3].set_index("Run")
d = fixed_k.loc["D"]
comparison = []
for run in ["A", "B", "C"]:
    other = fixed_k.loc[run]
    comparison.append({
        "D compared with": run,
        "Chunk-count change": f"{(d['Chunks']/other['Chunks']-1):+.1%}",
        "MRR change": f"{(d['MRR@k']/other['MRR@k']-1):+.1%}",
        "Accuracy change": f"{int(d['Answer quality (/9)']-other['Answer quality (/9)']):+d} correct",
        "Response-time change": f"{(d['Avg total response (s)']/other['Avg total response (s)']-1):+.1%}",
    })
pd.DataFrame(comparison)

  D compared with Chunk-count change MRR change Accuracy change Response-time change
0               A              -42.2%    +100.0%      +1 correct              +108.2%
1               B              +10.9%     +33.4%      +2 correct               +51.1%
2               C              +57.4%     +28.0%      +2 correct               +96.3%

## 5. Selected configuration and justification

### Selected: Configuration D — chunk size 800, overlap 200, top-k 3

1. **Best answer quality:** 3/9, versus 1–2/9 for every other run.
2. **Best ranking quality:** MRR@3 = 0.5926, the highest result. This means relevant evidence appeared earlier in the retrieved list more consistently.
3. **Controlled evidence for overlap:** compared with B (same 800-character size and top-k 3), increasing overlap from 100 to 200 improved MRR from 0.4444 to 0.5926 and accuracy from 1/9 to 3/9. The cost was 75 additional chunks and 0.291 seconds more average response time.
4. **Practical corpus size:** 762 chunks is small enough for the project’s Chroma collection and only 10.9% more than B.
5. **Trade-off accepted:** C was fastest to index and A responded faster, while E had the best Recall@k. For this medical-information RAG system, getting the correct supporting passage near the top and improving answer accuracy outweighs the sub-second latency increase.

This selection matches the production configuration in `backend/rag/config.py`.

## 6. Limitations

- The test set has 9 questions and one three-volume corpus; results should not be generalized beyond this dataset without more tests.
- Strict answer quality is low overall (maximum 3/9), indicating generation/re-ranking work is still needed.
- Timings depend on shared Colab hardware and should be treated comparatively, not as service-level guarantees.
- E changes top-k as well as retaining B's chunking, so it is a sensitivity control, not a pure chunking comparison.
- Character counts are not tokenizer token counts; the 1200-character run produced one 515-token warning with a 512-token model window.

In [ ]:
# Optional: export the recorded tables from this notebook.
output_path = Path("GroupApex_Chunking_Experiments_Results_rerun.xlsx")
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="Config Comparison", index=False)
    question_outputs_df.to_excel(writer, sheet_name="Question Outputs", index=False)
    pd.DataFrame(TESTS).to_excel(writer, sheet_name="Test Set", index=False)
print(f"Saved {output_path.resolve()}")